# 06 · Hechos de ventas y medidas

**Objetivo:** resolver NK a SK, respetar el grano y distinguir medidas aditivas.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [1]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Conexiones establecidas correctamente.
Servidor: localhost\SQLEXPRESS02
Origen: WideWorldImporters
Destino: WideWorldImportersDW2


## Una línea, un hecho

El identificador único conceptual es `InvoiceLineID`. La cabecera aporta
cliente, fecha y vendedor; el detalle aporta producto y medidas.

No duplicar montos de cabecera en cada línea. No sumar precios unitarios.
Este ejemplo usa dimensiones estáticas pequeñas; la búsqueda temporal SCD
se explica por separado en 05.


In [2]:
import pandas as pd
from decimal import Decimal

ventas = pd.DataFrame([
    {"linea_id": 1, "factura_id": 100, "cliente_nk": 1, "producto_nk": 10,
     "cantidad": 2, "total": Decimal("24.20"), "impuesto": Decimal("4.20")},
    {"linea_id": 2, "factura_id": 101, "cliente_nk": 1, "producto_nk": 20,
     "cantidad": -1, "total": Decimal("-6.05"), "impuesto": Decimal("-1.05")},
])
clientes = pd.DataFrame([{"cliente_nk": 1, "cliente_sk": 101}])
productos = pd.DataFrame([{"producto_nk": 10, "producto_sk": 201},
                         {"producto_nk": 20, "producto_sk": 202}])
hechos = ventas.merge(clientes, on="cliente_nk", how="left", validate="many_to_one")
hechos = hechos.merge(productos, on="producto_nk", how="left", validate="many_to_one")
if hechos[["cliente_sk", "producto_sk"]].isna().any().any():
    raise ValueError("Hay claves sin resolver; no publicar")
if hechos["linea_id"].duplicated().any():
    raise ValueError("El grano no es único")
hechos["neto"] = hechos["total"] - hechos["impuesto"]
display(hechos)


,linea_id,factura_id,cliente_nk,producto_nk,cantidad,total,impuesto,cliente_sk,producto_sk,neto
0,1,100,1,10,2,24.20,4.20,101,201,20.00
1,2,101,1,20,-1,-6.05,-1.05,101,202,-5.00


In [3]:
resumen = {
    campo: sum(hechos[campo], Decimal("0.00"))
    for campo in ["neto", "impuesto", "total"]
}
print(resumen)
print("Filas de hechos:", len(hechos))


{'neto': Decimal('15.00'), 'impuesto': Decimal('3.15'), 'total': Decimal('18.15')}
Filas de hechos: 2


## Resultado esperado y aplicación real

Dos hechos: neto **15.00**, impuesto **3.15**, total **18.15**.
La nota de crédito se conserva, no se descarta por ser negativa.

En un DW real se cargarían también `vendedor_sk`, `ciudad_sk` e `id_tiempo`.
Aquí se muestran dos dimensiones para hacer legible el mecanismo.

Reejecutar una carga no debe insertar otra vez el mismo detalle. Si cambia un
importe de un ID existente debe existir una política de actualización auditada
o ajuste; filtrar solo IDs nuevos ocultaría correcciones.


## Actividad de discusión

¿Por qué no agregar a nivel factura antes de resolver producto? **Respuesta:** se pierde el grano y ya no se pueden atribuir correctamente sus medidas.